# Data Collection Module
## Stock Prices + Financial Statements (S&P 500)

Collects:
- **Stock Prices** — OHLCV data via `yfinance` (batch download, 2015–present)
- **Financial Statements** — Income, Balance Sheet, Cash Flow per ticker (parallel fetch)
- **Macro Indicators** — Major indices, VIX, yields, FX, gold, oil

Set `FULL_SP500 = False` and use `FOCUS_TICKERS` for faster dev/testing runs.
Output saved to `data/raw/`.

In [ ]:
import yfinance as yf
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import io
import os

# --- CONFIG ---
FULL_SP500 = True   # Set False to use FOCUS_TICKERS only (faster for dev/testing)
FOCUS_TICKERS = [
    "MMM","AOS","ABT","ABBV","ACN","ADBE","AMD","AES","AFL","A","APD","ABNB",
    "AKAM","ALB","ARE","ALGN","ALLE","LNT","ALL","GOOGL","GOOG","MO","AMZN",
    "AMCR","AEE","AEP","AXP","AIG","AMT","AWK","AMP","AME","AMGN","APH","ADI",
    "ANSS","AON","APA","AAPL","AMAT","APTV","ACGL","ADM","ANET","AJG","AIZ",
    "T","ATO","ADSK","ADP","AZO","AVB","AVY","AXON","BKR","BALL","BAC","BAX",
    "BDX","BRK-B","BBY","TECH","BIIB","BLK","BX","BK","BA","BKNG","BWA","BSX",
    "BMY","AVGO","BR","BRO","BF-B","BLDR","BG","BXP","CHRW","CDNS","CZR","CPT",
    "CPB","COF","CAH","KMX","CCL","CARR","CAT","CBOE","CBRE","CDW","CE","COR",
    "CNC","CNP","CF","CRL","SCHW","CHTR","CVX","CMG","CB","CHD","CI","CINF",
    "CTAS","CSCO","C","CFG","CLX","CME","CMS","KO","CTSH","CL","CMCSA","CAG",
    "COP","ED","STZ","CEG","COO","CPRT","GLW","CPAY","CTVA","CSGP","COST","CTRA",
    "CRWD","CCI","CSX","CMI","CVS","DHR","DRI","DVA","DAY","DECK","DE","DELL",
    "DAL","DVN","DXCM","FANG","DLR","DFS","DG","DLTR","D","DPZ","DOV","DOW",
    "DHI","DTE","DUK","DD","EMN","ETN","EBAY","ECL","EIX","EW","EA","ELV",
    "EMR","ENPH","ETR","EOG","EPAM","EQT","EFX","EQIX","EQR","ERIE","ESS","EL",
    "EG","EVRG","ES","EXC","EXPE","EXPD","EXR","XOM","FFIV","FDS","FICO","FAST",
    "FRT","FDX","FIS","FITB","FSLR","FE","FI","FMC","F","FTNT","FTV","FOXA",
    "FOX","BEN","FCX","GRMN","IT","GE","GEHC","GEV","GEN","GNRC","GD","GIS",
    "GM","GPC","GILD","GPN","GL","GDDY","GS","HAL","HIG","HAS","HCA","DOC",
    "HSIC","HSY","HES","HPE","HLT","HOLX","HD","HON","HRL","HST","HWM","HPQ",
    "HUBB","HUM","HBAN","HII","IBM","IEX","IDXX","ITW","INCY","IR","PODD","INTC",
    "ICE","IFF","IP","IPG","INTU","ISRG","IVZ","INVH","IQV","IRM","JBHT","JBL",
    "JKHY","J","JNJ","JCI","JPM","JNPR","K","KVUE","KDP","KEY","KEYS","KMB",
    "KIM","KMI","KKR","KLAC","KHC","KR","LHX","LH","LRCX","LW","LVS","LDOS",
    "LEN","LIN","LYV","LKQ","LMT","L","LOW","LULU","LYB","MTB","MPC","MKTX",
    "MAR","MMC","MLM","MAS","MA","MTCH","MKC","MCD","MCK","MDT","MRK","META",
    "MET","MTD","MGM","MCHP","MU","MSFT","MAA","MRNA","MHK","MOH","TAP","MDLZ",
    "MPWR","MNST","MCO","MS","MOS","MSI","MSCI","NDAQ","NTAP","NFLX","NEM","NWSA",
    "NWS","NEE","NKE","NI","NDSN","NSC","NTRS","NOC","NCLH","NRG","NUE","NVDA",
    "NVR","NXPI","ORLY","OXY","ODFL","OMC","ON","OKE","ORCL","OTIS","PCAR","PKG",
    "PLTR","PANW","PARA","PH","PAYX","PAYC","PYPL","PNR","PEP","PFE","PCG","PM",
    "PSX","PNW","PNC","POOL","PPG","PPL","PFG","PG","PGR","PLD","PRU","PEG",
    "PTC","PSA","PHM","QRVO","PWR","QCOM","DGX","RL","RJF","RTX","O","REG","REGN",
    "RF","RSG","RMD","RVTY","ROK","ROL","ROP","ROST","RCL","SPGI","CRM","SBAC",
    "SLB","STX","SRE","NOW","SHW","SPG","SWKS","SJM","SW","SNA","SOLV","SO",
    "LUV","SWK","SBUX","STT","STLD","STE","SYK","SMCI","SYF","SNPS","SYY","TMUS",
    "TROW","TTWO","TPR","TRGP","TGT","TEL","TDY","TER","TSLA","TXN","TPL","TXT",
    "TMO","TJX","TSCO","TT","TDG","TRV","TRMB","TFC","TYL","TSN","USB","UBER",
    "UDR","ULTA","UNP","UAL","UPS","URI","UNH","UHS","VLO","VTR","VLTO","VRSN",
    "VRSK","VZ","VRTX","VTRS","VICI","V","VST","VMC","WRB","GWW","WAB","WBA",
    "WMT","DIS","WBD","WM","WAT","WEC","WFC","WELL","WST","WDC","WY","WSM",
    "WMB","WTW","WDAY","WYNN","XEL","XYL","YUM","ZBRA","ZBH","ZTS"
]
START_DATE = "2015-01-01"
END_DATE   = "2026-01-01"
OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

tickers = FOCUS_TICKERS
print(f"Using {len(tickers)} tickers  |  Range: {START_DATE} → {END_DATE}")

# --- 1. BATCH DOWNLOAD PRICES ---
print("\nFetching prices (batched)...")

BATCH_SIZE = 50
price_frames = []

for i in range(0, len(tickers), BATCH_SIZE):

    batch = tickers[i:i + BATCH_SIZE]

    print(
        f"Downloading batch "
        f"{i//BATCH_SIZE + 1}/"
        f"{(len(tickers)-1)//BATCH_SIZE + 1}"
    )

    try:

        batch_raw = yf.download(
            batch,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,
            progress=False,
            threads=False,
            group_by="ticker"
        )

        if batch_raw.empty:
            print("  Empty batch")
            continue

        if isinstance(batch_raw.columns, pd.MultiIndex):

            batch_frames = []

            for ticker in batch:

                try:

                    if ticker not in batch_raw.columns.levels[0]:
                        continue

                    df = batch_raw[ticker].copy()

                    if df.empty:
                        continue

                    df = df.reset_index()
                    df["Ticker"] = ticker

                    batch_frames.append(df)

                except Exception as e:
                    print(f"  Failed {ticker}: {e}")

            if batch_frames:

                batch_df = pd.concat(
                    batch_frames,
                    ignore_index=True
                )

                price_frames.append(batch_df)

        else:

            df = batch_raw.reset_index()

            if len(batch) == 1:
                df["Ticker"] = batch[0]

            price_frames.append(df)

    except Exception as e:
        print(f"Batch failed: {e}")

# Combine all batches
if not price_frames:
    raise ValueError("No price data downloaded")

price_df = pd.concat(
    price_frames,
    ignore_index=True
)

print(price_df.columns)

# Ensure required columns exist
required_cols = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Ticker"
]

missing = [
    c for c in required_cols
    if c not in price_df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

# Remove invalid rows
price_df = price_df.dropna(
    subset=["Close"]
)

# Memory optimisation
for col in ["Open", "High", "Low", "Close"]:

    price_df[col] = pd.to_numeric(
        price_df[col],
        errors="coerce"
    ).astype("float32")

price_df["Volume"] = pd.to_numeric(
    price_df["Volume"],
    errors="coerce"
).astype("Int64")

print(
    f"Price data ready: "
    f"{price_df.shape[0]:,} rows | "
    f"{price_df['Ticker'].nunique()} tickers"
)

# --- 2. PARALLEL FINANCIAL STATEMENTS ---
def fetch_financials(ticker: str) -> dict | None:
    """Fetch income statement, balance sheet, and cash flow for one ticker."""
    try:
        stock = yf.Ticker(ticker)
        return {
            "ticker":   ticker,
            "income":   stock.financials,
            "balance":  stock.balance_sheet,
            "cashflow": stock.cashflow,
        }
    except Exception:
        return None


def reshape(df: pd.DataFrame, ticker: str, name: str) -> pd.DataFrame | None:
    """Pivot a wide financial DF (items × dates) into long format with Ticker column."""
    if df is None or df.empty:
        return None
    df = df.T.copy()
    df.index.name = "Date"
    df = df.reset_index()
    df["Ticker"]    = ticker
    df["Statement"] = name
    return df


print(f"\nFetching financials for {len(tickers)} tickers (8 parallel workers)...")
print("Note: full S&P 500 may take 10–20 minutes due to API rate limits.\n")

all_income, all_balance, all_cashflow, failed = [], [], [], []

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(fetch_financials, t): t for t in tickers}

    for i, future in enumerate(as_completed(futures), 1):
        ticker = futures[future]
        if i % 50 == 0 or i == len(tickers):
            print(f"  Progress: {i}/{len(tickers)} ({100 * i // len(tickers)}%)")

        res = future.result()
        if res is None:
            failed.append(ticker)
            continue

        inc = reshape(res["income"],   ticker, "income")
        bal = reshape(res["balance"],  ticker, "balance")
        cf  = reshape(res["cashflow"], ticker, "cashflow")

        if inc is not None: all_income.append(inc)
        if bal is not None: all_balance.append(bal)
        if cf  is not None: all_cashflow.append(cf)

if failed:
    n = len(failed)
    sample = ", ".join(failed[:10]) + ("..." if n > 10 else "")
    print(f"\n[WARN] {n} tickers failed: {sample}")

# --- 3. CONCATENATE ---
print("\nConcatenating statements...")
income_df   = pd.concat(all_income,   ignore_index=True) if all_income   else pd.DataFrame()
balance_df  = pd.concat(all_balance,  ignore_index=True) if all_balance  else pd.DataFrame()
cashflow_df = pd.concat(all_cashflow, ignore_index=True) if all_cashflow else pd.DataFrame()

# --- 4. SAVE ---
print("Saving raw files...")
price_df.to_csv(   f"{OUTPUT_DIR}/sp500_prices.csv",   index=False)
income_df.to_csv(  f"{OUTPUT_DIR}/sp500_income.csv",   index=False)
balance_df.to_csv( f"{OUTPUT_DIR}/sp500_balance.csv",  index=False)
cashflow_df.to_csv(f"{OUTPUT_DIR}/sp500_cashflow.csv", index=False)

print(f"""
Done! Raw files saved to {OUTPUT_DIR}/
  sp500_prices.csv    — {price_df.shape[0]:,} rows
  sp500_income.csv    — {income_df.shape[0]:,} rows
  sp500_balance.csv   — {balance_df.shape[0]:,} rows
  sp500_cashflow.csv  — {cashflow_df.shape[0]:,} rows
""")

Using 501 tickers  |  Range: 2015-01-01 → 2026-01-01

Fetching prices (batched)...


Failed to get ticker 'APTV' reason: Failed to perform, curl: (28) Connection timed out after 30001 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$AMZN: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-01-01)


## Macro & Market Indicators

Collects major market indices, volatility index, treasury yield, USD index, gold, and crude oil.
Output saved to `data/raw/macro_market.csv`.

In [ ]:
import yfinance as yf
import pandas as pd
import os

OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Macro/market tickers with human-readable names
MACRO_TICKERS = {
    "^GSPC":    "S&P 500",
    "^DJI":     "Dow Jones",
    "^IXIC":    "NASDAQ",
    "^VIX":     "VIX (Fear Index)",
    "^TNX":     "10Y Treasury Yield",
    "DX-Y.NYB": "US Dollar Index",
    "GC=F":     "Gold Futures",
    "CL=F":     "Crude Oil WTI",
}

print("Fetching macro & market data...")
macro_raw = yf.download(
    list(MACRO_TICKERS.keys()),
    start="2020-01-01",
    end="2026-01-01",
    group_by="ticker",
    auto_adjust=False,
    progress=False
)

# Reshape to long format — yfinance ≥1.0 MultiIndex: names=['Price','Ticker']
if isinstance(macro_raw.columns, pd.MultiIndex):
    ticker_level = macro_raw.columns.names.index("Ticker") if "Ticker" in macro_raw.columns.names else 1
    macro_df = macro_raw.stack(level=ticker_level, future_stack=True).reset_index()
    if "Ticker" not in macro_df.columns:
        extra = [c for c in macro_df.columns if c not in ("Date", "Open", "High", "Low", "Close", "Volume", "Adj Close")]
        if extra:
            macro_df.rename(columns={extra[0]: "Ticker"}, inplace=True)
else:
    macro_df = macro_raw.reset_index()
    macro_df["Ticker"] = list(MACRO_TICKERS.keys())[0]

# Add human-readable name
macro_df["Name"] = macro_df["Ticker"].map(MACRO_TICKERS)

macro_df.to_csv(f"{OUTPUT_DIR}/macro_market.csv", index=False)

print(f"Macro data: {macro_df.shape[0]:,} rows, {macro_df['Ticker'].nunique()} instruments")
print(f"Saved to {OUTPUT_DIR}/macro_market.csv")
macro_df.tail()